# RAG Pipeline Practice Notebook

This notebook demonstrates a simple Retrieval-Augmented Generation (RAG) pipeline using:
- local text documents
- sentence splitting
- embeddings
- vector search
- an LLM for answer generation

This is designed for hands-on practice in Applied AI Engineering.

## 1. Install required packages

If packages are missing, install them before running the notebook.

In [ ]:
# Install dependencies
# !pip install sentence-transformers faiss-cpu langchain-openai openai python-dotenv
# If you prefer using Hugging Face embeddings locally, keep the package below:
# !pip install sentence-transformers

print('Package installation step ready. Uncomment and run if needed.')

## 2. Import libraries

In [ ]:
import os
from pathlib import Path
import textwrap

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Libraries imported successfully.')

## 3. Create a sample knowledge base

We create small documents that mimic a company knowledge base.

In [ ]:
documents = [
    "AI Engineering helps teams build production-ready AI systems using LLMs, APIs, tools, and workflows.",
    "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and adds it to the prompt before generation.",
    "Vector databases store embeddings so relevant content can be searched by similarity instead of exact keyword matching.",
    "LangChain and LangGraph are used to orchestrate prompts, tools, memory, and multi-step workflows in AI systems.",
    "Guardrails protect AI applications from prompt injection, unsafe outputs, and sensitive data leaks.",
    "AI agents can use tools, access APIs, and perform multi-step actions based on user goals.",
    "Monitoring AI systems involves tracking latency, cost, token usage, answer quality, and safety issues.",
    "A production-ready AI app should include evaluation, security checks, deployment, observability, and human review."
]

for i, doc in enumerate(documents, 1):
    print(f"Document {i}: {doc[:120]}\n")

## 4. Build embeddings for the documents

We use a lightweight local embedding model.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(documents, convert_to_numpy=True)
embeddings = np.array(embeddings).astype('float32')

print('Embedding shape:', embeddings.shape)
print('First embedding sample:', embeddings[0][:5])

## 5. Create FAISS vector index

FAISS allows fast similarity search.

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('FAISS index created with dimension:', dimension)

## 6. Retrieval function

This function searches for the most relevant chunks.

In [ ]:
def retrieve_top_k(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for idx in indices[0]:
        results.append(documents[int(idx)])
    return results

sample_query = 'What is RAG and why is it used in AI systems?'
retrieved = retrieve_top_k(sample_query, top_k=3)
print('Retrieved context:')
for i, doc in enumerate(retrieved, 1):
    print(f"{i}. {doc}")

## 7. Build a simple answer generator

For practice, we create a simple rule-based answer generator using retrieved context.
This can later be replaced by an LLM API like OpenAI or Azure OpenAI.

In [ ]:
def generate_answer(query):
    context_docs = retrieve_top_k(query, top_k=3)
    context = '\n'.join(context_docs)
    answer = f"Based on the retrieved knowledge:\n\n{context}\n\nAnswer to question: {query}"
    return answer

print(generate_answer(sample_query))

## 7. Build a simple answer generator

For practice, we create a simple rule-based answer generator using retrieved context.
This can later be replaced by an LLM API like OpenAI or Azure OpenAI.

In [ ]:
def generate_answer(query):
    context_docs = retrieve_top_k(query, top_k=3)
    context = '\n'.join(context_docs)
    answer = f"Based on the retrieved knowledge:\n\n{context}\n\nAnswer to question: {query}"
    return answer

print(generate_answer(sample_query))

## 8. Create a reusable RAG pipeline class

This gives a more production-like structure.

In [ ]:
class SimpleRAGPipeline:
    def __init__(self, documents, model_name='all-MiniLM-L6-v2'):
        self.documents = documents
        self.embedding_model = SentenceTransformer(model_name)
        self.embeddings = self.embedding_model.encode(documents, convert_to_numpy=True).astype('float32')
        self.index = faiss.IndexFlatL2(self.embeddings.shape[1])
        self.index.add(self.embeddings)

    def retrieve(self, query, top_k=3):
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True).astype('float32')
        distances, indices = self.index.search(query_embedding, top_k)
        return [self.documents[int(i)] for i in indices[0]]

    def answer(self, query):
        context = '\n'.join(self.retrieve(query, top_k=3))
        return f"Context:\n{context}\n\nQuestion: {query}\nAnswer: The system uses retrieved knowledge to ground the response."

rag = SimpleRAGPipeline(documents)
print(rag.answer('What are the key ideas behind AI agents?'))

## 9. Add a more realistic prompt-based answer

This version creates a prompt that includes the retrieved context.

In [ ]:
def build_prompt(query, context_docs):
    context = '\n'.join(f'- {doc}' for doc in context_docs)
    prompt = f"
"
You are a helpful AI assistant. Use the context below to answer the question.
If the answer is not in the context, say you do not have enough information.

Context:
{context}

Question: {query}
Answer:


"""
    return prompt

query = 'How do guardrails improve AI applications?'
context_docs = rag.retrieve(query, top_k=3)
prompt = build_prompt(query, context_docs)

print(prompt)

## 10. Practice exercises

Try the following tasks:
1. Add more documents and test retrieval quality.
2. Try different chunk sizes and compare results.
3. Use a real LLM API (OpenAI or Azure OpenAI) instead of the static answer generator.
4. Store results in a structured JSON file.
5. Add a web API using FastAPI so the RAG system is exposed as an app.
6. Add a guardrail check before returning the response.

## 11. Real-world extension: Connect to a real LLM API

Example idea:
- Use OpenAI / Azure OpenAI API
- Send the prompt to the model
- Return final generated answer
- Add logging and output validation

In [ ]:
# Example for future use:
# from openai import OpenAI
# client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# response = client.chat.completions.create(
#     model='gpt-4o-mini',
#     messages=[{'role':'user','content': prompt}]
# )
# print(response.choices[0].message.content)

print('LLM API integration example is ready to be enabled.')

## 12. Final takeaway

This notebook is a working introduction to RAG systems:
- create knowledge documents
- embed them
- index them with vector search
- retrieve relevant chunks
- generate grounded answers

This is a foundational AI Engineering skill for modern AI systems.